# 01 — 场景需求总览

**目标**：快速了解 10 个早晚高峰场景的订单量、供需差异概况，为后续地图可视化提供全局视角。

**数据来源**：
- `data/processed/scenario_order_counts.csv` — 每个场景的订单量
- `data/processed/scenario_grid_demand_200m.parquet` — 200m 网格供需明细

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('库加载完成')

In [ ]:
# 读取场景订单量汇总
order_counts = pd.read_csv('../data/processed/scenario_order_counts.csv')

# 读取 200m 网格需求（核心分析尺度）
demand_200m = pd.read_parquet('../data/processed/scenario_grid_demand_200m.parquet')

# 读取数据质量摘要
quality = pd.read_csv('../data/processed/data_quality_summary.csv')

# 将 peak_type 换成中文标签
peak_label = {'am_peak': '早高峰 (07-10h)', 'pm_peak': '晚高峰 (17-20h)'}
order_counts['peak_label'] = order_counts['peak_type'].map(peak_label)

print(f'场景数: {order_counts["scenario_id"].nunique()}')
print(f'200m 网格需求总行数: {len(demand_200m):,}')
print(f'订单量范围: {order_counts["orders"].min()} ~ {order_counts["orders"].max()}')
order_counts

## 1. 十个场景订单量对比

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = ['#4C72B0' if pt == 'am_peak' else '#DD8452' for pt in order_counts['peak_type']]
bars = ax.bar(range(len(order_counts)), order_counts['orders'], color=colors, edgecolor='white', linewidth=0.5)

ax.set_xticks(range(len(order_counts)))
ax.set_xticklabels(order_counts['scenario_id'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('订单数', fontsize=12)
ax.set_title('十个早晚高峰场景 — 福田区共享单车订单量', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# 在柱子上标数值
for bar, val in zip(bars, order_counts['orders']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200, f'{val:,}',
            ha='center', va='bottom', fontsize=8)

# 图例
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='早高峰'), Patch(facecolor='#DD8452', label='晚高峰')]
ax.legend(handles=legend_elements, loc='upper right')

ax.set_ylim(0, order_counts['orders'].max() * 1.2)
plt.tight_layout()
plt.savefig('../outputs/figures/01_scenario_order_counts.png', dpi=150, bbox_inches='tight')
plt.show()

> ⚠️ **注意**：周一早高峰（20210510_am_peak）订单量异常高（26,664），远超其他场景（3,758 ~ 5,630）。这可能是因为抽样时该窗口数据密度更高，或周一通勤需求确实更大。后续分析时需要注意这一差异。

## 2. 早晚高峰分工作日对比

In [ ]:
pivot = order_counts.pivot_table(values='orders', index='date', columns='peak_type', aggfunc='sum')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(pivot))
width = 0.35

bars1 = ax.bar(x - width/2, pivot.get('am_peak', [0]*len(pivot)), width,
               color='#4C72B0', edgecolor='white', label='早高峰')
bars2 = ax.bar(x + width/2, pivot.get('pm_peak', [0]*len(pivot)), width,
               color='#DD8452', edgecolor='white', label='晚高峰')

ax.set_xticks(x)
ax.set_xticklabels(['5/10 周一', '5/11 周二', '5/12 周三', '5/13 周四', '5/14 周五'], fontsize=10)
ax.set_ylabel('订单数', fontsize=12)
ax.set_title('工作日早晚高峰订单量对比', fontsize=14, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('../outputs/figures/01_am_pm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 各场景网格级供需差异统计

200m 网格尺度下，每个场景有多少网格缺车（shortage > 0）、多少网格富余（surplus > 0）？

In [ ]:
# 按场景统计
grid_stats = demand_200m.groupby('scenario_id').agg(
    total_grids=('grid_id', 'nunique'),
    shortage_grids=('shortage', lambda x: (x > 0).sum()),
    surplus_grids=('surplus', lambda x: (x > 0).sum()),
    total_shortage=('shortage', 'sum'),
    total_surplus=('surplus', 'sum'),
    total_departures=('departures', 'sum'),
    total_arrivals=('arrivals', 'sum')
).reset_index()

grid_stats['net_outflow_total'] = grid_stats['total_departures'] - grid_stats['total_arrivals']
grid_stats

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(grid_stats))
width = 0.35

ax.bar(x - width/2, grid_stats['shortage_grids'], width, color='#C44E52', edgecolor='white', label='短缺网格数')
ax.bar(x + width/2, grid_stats['surplus_grids'], width, color='#55A868', edgecolor='white', label='富余网格数')

ax.set_xticks(x)
ax.set_xticklabels(grid_stats['scenario_id'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('网格数量', fontsize=12)
ax.set_title('各场景短缺 vs 富余网格数量（200m 尺度）', fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/01_surplus_shortage_grids.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 总短缺量和总富余量对比
fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(x - width/2, grid_stats['total_shortage'], width, color='#C44E52', edgecolor='white', label='总短缺量（辆）')
ax.bar(x + width/2, grid_stats['total_surplus'], width, color='#55A868', edgecolor='white', label='总富余量（辆）')

ax.set_xticks(x)
ax.set_xticklabels(grid_stats['scenario_id'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('单车数量', fontsize=12)
ax.set_title('各场景总短缺量 vs 总富余量（200m 尺度）', fontsize=14, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('../outputs/figures/01_surplus_shortage_bikes.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 数据清洗摘要

In [ ]:
quality

---

## 关键发现

1. **周一早高峰异常值**：20210510_am_peak 有 26,664 条订单，是其他场景的 5-7 倍，需要在后续分析中特别标注。
2. **早晚高峰差异**：除周一外，早高峰订单量普遍略低于晚高峰。
3. **供需总量平衡**：总短缺量 ≈ 总富余量，说明系统整体车辆守恒，问题核心在于**空间分布不均**。
4. **数据清洗**：原始 291,000 → 清洗后 69,357（主要删除了不在福田区的订单 217,022 条）。

## 下一步

→ `02_surplus_shortage_map.ipynb` — 在地图上可视化每个场景的短缺/富余空间分布